# Module 8: Delta Lake & AWS Migration Patterns

**Context**: Preparing for Databricks → AWS migration projects

## Topics
1. Delta Lake fundamentals (ACID on data lakes)
2. Delta Lake operations (MERGE, TIME TRAVEL, OPTIMIZE)
3. Databricks vs AWS Glue/EMR differences
4. Migration patterns & compatibility
5. S3 integration patterns

In [ ]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    S3_RAW = "s3a://sparkling-data-test/data/raw"
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Module08-DeltaLake").master("local[*]").config("spark.sql.shuffle.partitions", "8").config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0").config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension").config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog").getOrCreate()
    MODE = 'local'
    S3_RAW = None
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

---
## 1. Delta Lake Fundamentals

### Why Delta Lake?
| Problem | Parquet | Delta Lake |
|---------|---------|------------|
| ACID transactions | ❌ | ✅ |
| Schema enforcement | Partial | ✅ Full |
| Time travel | ❌ | ✅ |
| MERGE (upsert) | ❌ | ✅ |
| Small file compaction | Manual | OPTIMIZE |
| Concurrent writes | Unsafe | ✅ Safe |

In [ ]:
# ── Load data (S3 Parquet or local CSV) ──
transactions = spark.read.parquet(f"{S3_RAW}/transactions") if MODE == "databricks" else spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)

print(f"Loaded {transactions.count():,} transactions")
delta_txn_path = str(DELTA_PATH / "transactions")
transactions.write.format("delta").mode("overwrite").save(delta_txn_path)
print(f"✅ Written to Delta: {delta_txn_path}")

In [ ]:
# Read Delta table
delta_df = spark.read.format("delta").load(delta_txn_path)
delta_df.show(5)

---
## 2. Delta Lake MERGE (Upsert)

**Key for SCD & incremental loads** - replaces complex INSERT/UPDATE logic.

In [ ]:
from delta.tables import DeltaTable

# Create Delta table reference
delta_table = DeltaTable.forPath(spark, delta_txn_path)

# Simulate updates (10 modified transactions)
updates = transactions.limit(10).withColumn("amount", col("amount") * 2).withColumn("status", lit("Updated"))
updates.show(3)

In [ ]:
# MERGE operation (upsert)
delta_table.alias("target").merge(
    updates.alias("source"),
    "target.txn_id = source.txn_id"
).whenMatchedUpdate(set={
    "amount": "source.amount",
    "status": "source.status"
}).whenNotMatchedInsertAll().execute()

print("✅ MERGE completed")

# Verify updates
spark.read.format("delta").load(delta_txn_path).filter(col("status") == "Updated").show()

---
## 3. Time Travel

Query historical versions - essential for debugging and auditing.

In [ ]:
# View table history
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

In [ ]:
# Read previous version (before MERGE)
old_version = spark.read.format("delta").option("versionAsOf", 0).load(delta_txn_path)
print(f"Version 0 count: {old_version.count():,}")

# Compare
current = spark.read.format("delta").load(delta_txn_path)
print(f"Current count: {current.count():,}")

---
## 4. OPTIMIZE & Z-ORDER

Compacts small files and optimizes for specific query patterns.

In [ ]:
# Optimize with Z-ORDER on frequently filtered columns
spark.sql(f"OPTIMIZE delta.`{delta_txn_path}` ZORDER BY (account_id, txn_datetime)")
print("✅ OPTIMIZE complete")

---
## 5. Databricks vs AWS: Key Differences

| Aspect | Databricks | AWS EMR/Glue |
|--------|------------|---------------|
| **Delta Lake** | Native, optimized | OSS Delta (same API) |
| **Cluster mgmt** | Managed pools | EMR: manual, Glue: serverless |
| **Notebooks** | Built-in | EMR Studio, SageMaker |
| **Catalog** | Unity Catalog | AWS Glue Data Catalog |
| **Cost model** | DBU-based | EC2/Glue DPU |
| **Delta features** | All (Photon, etc.) | OSS features only |

In [ ]:
# === Migration Pattern: Databricks → AWS EMR ===

# 1. Replace Databricks-specific imports
# Databricks: from databricks.sdk import ...
# AWS: Use boto3, awswrangler

# 2. S3 path format (same in both)
s3_path = "s3://your-bucket/delta/transactions"

# 3. Glue Catalog integration (AWS specific)
spark_glue = SparkSession.builder \
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3://your-bucket/warehouse") \
    .getOrCreate()

print("AWS config pattern shown (not executed - requires AWS credentials)")

---
## 6. Migration Checklist: Databricks → AWS

### Code Changes
- [ ] Replace `dbutils` with boto3/S3 clients
- [ ] Replace `display()` with `show()` or print
- [ ] Update Unity Catalog → Glue Catalog references
- [ ] Replace Databricks SQL endpoints → Athena/Redshift Serverless

### Delta Lake Compatibility
- [ ] Use OSS Delta Lake version matching Databricks
- [ ] Test MERGE, TIME TRAVEL operations
- [ ] OPTIMIZE works, but Photon optimizations won't transfer

### Infrastructure
- [ ] EMR clusters or Glue jobs for Spark
- [ ] Step Functions or Airflow for orchestration
- [ ] CloudWatch instead of Databricks Jobs UI

In [ ]:
# === Pattern: Write to S3 with Glue Catalog registration ===

def write_delta_to_aws(df, table_name, s3_path, partition_cols=None):
    """
    Write Delta table to S3 and register in Glue Catalog.
    Use this pattern for AWS migrations.
    """
    writer = df.write.format("delta").mode("overwrite")
    
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    
    # Write to S3
    writer.save(s3_path)
    
    # Register in Glue Catalog (if configured)
    # spark.sql(f"CREATE TABLE IF NOT EXISTS glue_catalog.db.{table_name} USING DELTA LOCATION '{s3_path}'")
    
    print(f"✅ Written to {s3_path}")

# Local test
write_delta_to_aws(transactions.limit(1000), "transactions_sample", str(DELTA_PATH / "txn_sample"))

---
## Practice: Migration Scenario

**Scenario**: Migrate a Databricks pipeline that:
1. Reads from Delta Lake daily partition
2. Applies SCD Type 2 to customer dimension
3. Writes aggregated metrics to reporting table

**Task**: Rewrite using AWS-compatible patterns

In [ ]:
spark.stop()